# Similarity, the chasm and supersession

Every build has a point $u_b$ in concept space. A review of build $r$ acts on build $b$ through their similarity

$$\sigma(b,r) = \chi(b,r)\,\exp\!\left(-\frac{\lVert u_b-u_r\rVert^2}{2h^2}\right), \qquad h^2 = \frac{\rho^2}{-2\ln\sigma_{\min}},$$

so that a build at distance $\rho$ (the reach) sits exactly at $\sigma_{\min}$. Three levels matter:

| level | meaning |
|---|---|
| $\sigma_{\min}$ | below: no conflict and no reach |
| $\sigma_{\text{full}}$ | at or above: the newer claim replaces the older outright, and reach counts as 1 |
| $\sigma_{\text{same}}$ | at or above: the reviewed build itself |

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Checkbox
from tri.api import TriConfig

cfg = TriConfig()
L, S = cfg.ledger, cfg.infer               # similarity levels and scorer constants of this version
h = L.reach / np.sqrt(-2 * np.log(L.sim_min))
print(f"reach {L.reach}, sim_min {L.sim_min}, sim_full {L.sim_full}, sim_same {L.sim_same}, h = {h:.3f}")

## The distance kernel

Drag the reach $\rho$: the levels stay where they are on the $\sigma$ axis, and only the distance at which they are crossed moves.

In [ ]:
def kernel(rho=float(L.reach), smin=float(L.sim_min)):
    hh = rho / np.sqrt(-2 * np.log(smin))
    d = np.linspace(0, 3 * rho, 400)
    s = np.exp(-d ** 2 / (2 * hh ** 2))
    fig, ax = plt.subplots(figsize=(7, 3.2))
    ax.plot(d, s, lw=2)
    for lv, name in ((smin, "min"), (L.sim_full, "full"), (L.sim_same, "same")):
        dl = hh * np.sqrt(-2 * np.log(lv))
        ax.axhline(lv, ls=":", c="grey"); ax.axvline(dl, ls=":", c="grey")
        ax.text(dl, 0.05, f" {name} ({lv}): d = {dl:.2f}", rotation=90, va="bottom", fontsize=8)
    ax.set(xlabel="distance in concept space", ylabel=r"$\sigma$", title=f"h = {hh:.3f}", ylim=(0, 1.05))
    plt.show()

interact(kernel, rho=FloatSlider(value=float(L.reach), min=0.4, max=3, step=0.05),
         smin=FloatSlider(value=float(L.sim_min), min=0.5, max=0.95, step=0.01));

## The chasm

Distance alone would let a review leak into a neighbouring archetype. The chasm factor stops it:

$$\chi = 1 - \beta\, a(u_b)\, a(u_r)\left(1-(1-\mathrm{TV})^{\gamma}\right),$$

where $\mathrm{TV}$ is the total variation distance between the archetype posteriors of the two points and $a$ is the locator activity. With $\beta = 1$ and both builds located in different families, $\chi = 0$: a border between archetypes is a chasm, not a slope.

Below is a toy 2-D concept space with two archetype families split along $x$. The review sits at the white cross; colour is its reach $\sigma$. Move the review toward the border and watch the reach stop at it, however close the other side is.

In [ ]:
def chasm(x0=-0.6, y0=0.0, sharp=6.0, gamma=float(cfg.locators.chasm_power), beta=1.0, activity=1.0):
    g = np.linspace(-3, 3, 241)
    X, Y = np.meshgrid(g, g)
    p = lambda x: 1 / (1 + np.exp(-sharp * x))          # posterior of family B along x
    tv = np.abs(p(X) - p(x0))                            # TV between two-family posteriors
    chi = 1 - beta * activity ** 2 * (1 - (1 - tv) ** gamma)
    s = chi * np.exp(-((X - x0) ** 2 + (Y - y0) ** 2) / (2 * h ** 2))
    fig, axs = plt.subplots(1, 2, figsize=(10, 4))
    im = axs[0].imshow(s, extent=(-3, 3, -3, 3), origin="lower", vmin=0, vmax=1, cmap="magma")
    axs[0].contour(X, Y, s, levels=[L.sim_min, L.sim_full], colors=["w", "c"], linewidths=1)
    axs[0].axvline(0, c="w", ls="--", lw=0.8); axs[0].plot(x0, y0, "w+", ms=14, mew=2)
    axs[0].set_title(r"$\sigma$ (white: $\sigma_{min}$, cyan: $\sigma_{full}$)"); plt.colorbar(im, ax=axs[0])
    axs[1].plot(g, s[120], label=r"$\sigma$ along $y=0$"); axs[1].plot(g, chi[120], "--", label=r"$\chi$")
    axs[1].axhline(L.sim_min, c="grey", ls=":"); axs[1].axvline(0, c="k", ls="--", lw=0.8)
    axs[1].legend(); axs[1].set_ylim(0, 1.05); axs[1].set_xlabel("x")
    plt.tight_layout(); plt.show()

interact(chasm, x0=FloatSlider(value=-0.6, min=-2.5, max=2.5, step=0.05), y0=FloatSlider(value=0, min=-2, max=2, step=0.05),
         sharp=FloatSlider(value=6, min=0.5, max=20, step=0.5), gamma=FloatSlider(value=6, min=1, max=12, step=0.5),
         beta=FloatSlider(value=1, min=0, max=1, step=0.05), activity=FloatSlider(value=1, min=0, max=1, step=0.05));

## Supersession: the newer review wins

A review gives one claim per item: explicit operations become add or remove claims, sidegrade marks become iffy claims and unmarked items become keep claims. Each claim starts at weight $w_c = 1$ (0.25 for a claim made before a patch changed its item, and for an unmarked item of a build tri made itself). When two claims on one item disagree, the older is scaled by

$$f(\sigma) = \begin{cases} 0 & \sigma \ge \sigma_{\text{full}} \\ \dfrac{\sigma_{\text{full}}-\sigma}{\sigma_{\text{full}}-\sigma_{\min}} & \sigma_{\min} \le \sigma < \sigma_{\text{full}} \\ 1 & \sigma < \sigma_{\min} \end{cases}$$

taking the smallest factor over all newer conflicting claims. A keep supersedes an older add or remove only at $\sigma \ge \sigma_{\text{same}}$, and a claim conditional on items the other build lacks does not conflict. Supersession is computed once when the review log is set; nothing is refitted.

In [ ]:
def f(s):
    return np.where(s >= L.sim_full, 0.0, np.where(s >= L.sim_min, (L.sim_full - s) / (L.sim_full - L.sim_min), 1.0))

def supersede(sigma=0.9, keep=False):
    s = np.linspace(0.6, 1, 400)
    lim = L.sim_same if keep else L.sim_full
    ff = np.where(s >= lim, 0.0, f(s)) if keep else f(s)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(s, ff, lw=2); ax.axvline(sigma, c="r")
    v = float(0.0 if sigma >= lim else f(np.array(sigma))) if not keep else float(0.0 if sigma >= lim else 1.0)
    ax.set(xlabel=r"$\sigma$ between the two reviewed builds", ylabel="weight left on the older claim",
           title=f"older claim keeps w = {v:.2f}" + (" (newer claim is a keep)" if keep else ""))
    plt.show()

interact(supersede, sigma=FloatSlider(value=0.9, min=0.6, max=1, step=0.005), keep=Checkbox(False));